In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [2]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

In [3]:
# 2.Define steps


def step_1(state: CrashState) -> CrashState:
    print("Step 1 executed")
    return {"step1": "done", "input": state["input"]}


def step_2(state: CrashState) -> CrashState:
    print(
        " Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)"
    )
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}


def step_3(state: CrashState) -> CrashState:
    print("Step 3 executed")
    return {"step3": "done"}

In [4]:
builder = StateGraph(CrashState)

builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

In [9]:
checkpointer = InMemorySaver()
workflow = builder.compile(checkpointer=checkpointer)

try:
    workflow.invoke(
        {"input": "test input"}, config={"configurable": {"thread_id": "thread-1"}}
    )  # Start the workflow with initial input
except KeyboardInterrupt:
    print("Workflow interrupted by user.")

Step 1 executed
 Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
Workflow interrupted by user.


In [10]:
workflow.get_state(
    {"configurable": {"thread_id": "thread-1"}}
)  # Get the current state of the workflow

StateSnapshot(values={'input': 'test input', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c24c6-e3c4-63ec-8001-e4a23b980523'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-10-07T12:41:34.794238+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c24c6-e3c1-6cdd-8000-c829fda6ac03'}}, tasks=(PregelTask(id='60375517-7c15-a8fa-85ee-7c4f95b6e324', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [13]:
# Rerun the workflow from the last saved state
workflow.invoke(
    None, config={"configurable": {"thread_id": "thread-1"}}
)  # Resume the workflow from the last saved state

 Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
Step 3 executed


{'input': 'test input', 'step1': 'done', 'step2': 'done', 'step3': 'done'}

In [14]:
list(workflow.get_state_history({"configurable": {"thread_id": "thread-1"}}))

[StateSnapshot(values={'input': 'test input', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c24cf-45e6-686f-8003-d6c90667e44a'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-10-07T12:45:19.832689+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c24cf-45e6-686e-8002-3929ea94c1ad'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'test input', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c24cf-45e6-686e-8002-3929ea94c1ad'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-07T12:45:19.832689+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c24c6-e3c4-63ec-8001-e4a23b980523'}}, tasks=(PregelTask(id='8c5e8207